# Generalized Model-Answer Filler — No-Prompt Keys, Deterministic Params, Consistent System Prompt

This notebook reads a prompts JSON (list or dict with `"outputs"` / `"items"` / `"data"`), calls a selected LLM for each prompt,
and writes a **new JSON** with the same top-level structure but a `model_answer` added for every entry.

It uses a single **SYSTEM_WRITER** across all providers and fixes caller-side parameters to reduce variation.
It does **not** prompt you for API keys and does **not** use environment variables — you paste keys in one place (3).

**Run order:**
1) Environment & Reproducibility  
2) Config (set your prompts file path)  
3) API Keys (bare-bones; paste strings only)  
4) System Writer (global, consistent)  
5) Providers (OpenAI / Anthropic / Gemini; keys passed directly)  
6) Load & Save utilities  
7) Filler (cache/resume/checkpoints)  
8) Run Filler  


In [ ]:
# 1) Environment & Reproducibility
import os, sys, platform, random, json, time
from datetime import datetime
from pathlib import Path

def _ver(mod):
    try:
        return getattr(mod, '__version__', 'unknown')
    except Exception:
        return 'unknown'

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import torch
except Exception:
    torch = None

ENV = {
    'python': sys.version.replace('\n',' '),
    'platform': platform.platform(),
    'versions': {
        'numpy': _ver(np), 'pandas': _ver(pd), 'torch': _ver(torch)
    },
    'timestamp': datetime.now().isoformat(timespec='seconds')
}
print('=== Environment ===')
print(json.dumps(ENV, indent=2))

# Deterministic seeds (caller side)
SEED = int(os.environ.get('VALENCE_GLOBAL_SEED','100'))
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
if np is not None:
    np.random.seed(SEED)
if torch is not None:
    try:
        torch.manual_seed(SEED)
        torch.use_deterministic_algorithms(True)
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
    except Exception as e:
        print('PyTorch determinism partial:', repr(e))
print('Seeds set to', SEED)


In [ ]:
# 2) Config & Paths — SET YOUR PROMPTS FILE HERE (generalized)

from dataclasses import dataclass
from typing import Dict, Any

@dataclass
class Config:
    base_dir: Path = Path().resolve()
    outputs_dir: Path = Path('outputs_gpt-4o')
    cache_dir: Path = Path('cache')
    seed: int = SEED

CFG = Config()
for d in [CFG.outputs_dir, CFG.cache_dir]:
    d.mkdir(parents=True, exist_ok=True)

# REQUIRED: point this to the JSON you want to process.
PROMPTS_PATH = Path('../examples/sample_prompts.json')

RUN_DIR = CFG.outputs_dir / 'json_filled'
RUN_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = CFG.cache_dir / 'json_filled_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def write_manifest(path: Path, llm_name: str, extras: Dict[str, Any]):
    manifest = {
        'llm': llm_name,
        'env': ENV,
        'cfg': {
            'base_dir': str(CFG.base_dir),
            'outputs_dir': str(CFG.outputs_dir),
            'cache_dir': str(CFG.cache_dir),
        },
        'input_file': str(PROMPTS_PATH),
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'extras': extras,
    }
    (path.parent / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Manifest written to:', path.parent / 'manifest.json')

print('Prompts path:', PROMPTS_PATH)
print('Run dir:', RUN_DIR)
print('Cache dir:', CACHE_DIR)


In [ ]:
# 3) API keys — read from the environment (or a .env file). Never hardcode a key here.
#    export OPENAI_API_KEY=... / ANTHROPIC_API_KEY=... / GOOGLE_API_KEY=...
#    Leave a provider's variable unset to skip that provider.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
GEMINI_API_KEY    = os.getenv("GOOGLE_API_KEY", "")

# Convenience dict consumed by get_provider() below.
LOCAL_KEYS = {
    "openai": OPENAI_API_KEY or None,
    "anthropic": ANTHROPIC_API_KEY or None,
    "gemini": GEMINI_API_KEY or None,
}


In [ ]:
# 4) SYSTEM_WRITER (exact text; keep consistent to avoid output variation)
SYSTEM_WRITER = (  # can edit this here to change the context the LLM's response
    "You are a well-informed writer."
    "Respond to each request to the best of your abilities. Please do not ask clarification questions."
)
print('SYSTEM_WRITER length:', len(SYSTEM_WRITER))


In [ ]:
# 5) Providers (OpenAI / Anthropic / Gemini) — direct keys only, deterministic params, retries
import time, hashlib, random
from typing import Optional

MAX_RETRIES = 5
BASE_BACKOFF = 1.5

def _backoff_sleep(attempt: int):
    delay = (BASE_BACKOFF ** attempt) + random.random()
    time.sleep(min(delay, 30.0))

class ProviderError(RuntimeError):
    pass

class BaseProvider:
    name: str = 'base'
    def available(self) -> bool:
        raise NotImplementedError
    def generate(self, prompt: str, *, temperature: float=0.0, max_tokens: int=512) -> str:
        raise NotImplementedError

class OpenAIProvider(BaseProvider):
    name = 'openai'
    def __init__(self, model='gpt-4o', api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            import openai
            self._client = openai.OpenAI(api_key=self.api_key) if self.api_key else None
        except Exception:
            self._client = None
    def available(self) -> bool:
        return self._client is not None
    def generate(self, prompt: str, *, temperature: float=0.0, max_tokens: int=512) -> str:
        if not self.available():
            raise ProviderError('OpenAI not available — provide OPENAI_API_KEY and install the openai SDK.')
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                resp = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {'role':'system','content': SYSTEM_WRITER},
                        {'role':'user','content': prompt},
                    ],
                    temperature=0.0,
                    top_p=1,
                    frequency_penalty=0,
                    presence_penalty=0,
                    max_tokens=max_tokens,
                )
                return resp.choices[0].message.content
            except Exception as e:
                last_err = e
                print(f'[OpenAI] retry {attempt}/{MAX_RETRIES} after error: {e}')
                _backoff_sleep(attempt)
        raise ProviderError(f'OpenAI failed after retries: {last_err!r}')

class AnthropicProvider(BaseProvider):
    name = 'anthropic'
    def __init__(self, model='claude-3-5-sonnet-latest', api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            import anthropic
            self._client = anthropic.Anthropic(api_key=self.api_key) if self.api_key else None
        except Exception:
            self._client = None
    def available(self) -> bool:
        return self._client is not None
    def generate(self, prompt: str, *, temperature: float=0.0, max_tokens: int=512) -> str:
        if not self.available():
            raise ProviderError('Anthropic not available — provide ANTHROPIC_API_KEY and install the anthropic SDK.')
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                msg = self._client.messages.create(
                    model=self.model,
                    system=SYSTEM_WRITER,
                    messages=[{'role':'user','content': prompt}],
                    temperature=0.0,
                    max_tokens=max_tokens,
                )
                chunks = getattr(msg, 'content', [])
                text = ''.join([c.text for c in chunks if hasattr(c,'text')])
                return text or '[Empty Anthropic response]'
            except Exception as e:
                last_err = e
                print(f'[Anthropic] retry {attempt}/{MAX_RETRIES} after error: {e}')
                _backoff_sleep(attempt)
        raise ProviderError(f'Anthropic failed after retries: {last_err!r}')

class GeminiProvider(BaseProvider):
    name = 'gemini'
    def __init__(self, model='gemini-1.5-pro', api_key: Optional[str]=None):
        self.model = model
        self.api_key = api_key
        try:
            import google.generativeai as genai
            if self.api_key:
                genai.configure(api_key=self.api_key)
                try:
                    self._model = genai.GenerativeModel(self.model, system_instruction=SYSTEM_WRITER)
                except TypeError:
                    self._model = genai.GenerativeModel(self.model)
            else:
                self._model = None
        except Exception:
            self._model = None
    def available(self) -> bool:
        return self._model is not None
    def generate(self, prompt: str, *, temperature: float=0.0, max_tokens: int=512) -> str:
        if not self.available():
            raise ProviderError('Gemini not available — provide GEMINI_API_KEY and install google-generativeai.')
        last_err = None
        for attempt in range(1, MAX_RETRIES+1):
            try:
                content = prompt
                try:
                    _ = self._model.generate_content
                except Exception:
                    content = SYSTEM_WRITER + '\\n\\n' + prompt
                resp = self._model.generate_content(
                    content,
                    generation_config={
                        'temperature': 0.0,
                        'top_p': 1,
                        'max_output_tokens': max_tokens,
                    }
                )
                return getattr(resp, 'text', None) or (getattr(resp, 'candidates', [{}])[0].get('content', {}).get('parts', [{}])[0].get('text', '')) or '[Empty Gemini response]'
            except Exception as e:
                last_err = e
                print(f'[Gemini] retry {attempt}/{MAX_RETRIES} after error: {e}')
                _backoff_sleep(attempt)
        raise ProviderError(f'Gemini failed after retries: {last_err!r}')

def get_provider(name: str) -> BaseProvider:
    key = (name or '').lower()
    if key in ('openai','gpt','gpt-4o','gpt4o'):
        return OpenAIProvider(api_key=OPENAI_API_KEY or None)
    if key in ('anthropic','claude','sonnet','claude-3-5-sonnet'):
        return AnthropicProvider(api_key=ANTHROPIC_API_KEY or None)
    if key in ('gemini','google','gemini-1.5-pro'):
        return GeminiProvider(api_key=GEMINI_API_KEY or None)
    raise ValueError(f'Unknown provider: {name}')


In [ ]:
# 6) Load & Save utilities (robust; preserves top-level shape)
import json
from json import JSONDecodeError
from typing import List, Dict, Any, Tuple

def read_json_like(path: Path):
    txt = path.read_text(encoding='utf-8-sig')
    try:
        return json.loads(txt)
    except JSONDecodeError:
        # NDJSON fallback (one JSON object per line)
        data = []
        for ln_no, line in enumerate(txt.splitlines(), 1):
            s = line.strip()
            if not s:
                continue
            try:
                data.append(json.loads(s))
            except JSONDecodeError as e:
                raise JSONDecodeError(f'NDJSON parse error on line {ln_no}: {e.msg}', e.doc, e.pos)
        return data


def peel_to_list(data: Any) -> Tuple[List[Any], str]:
    """Return (list_ref, container_key) where container_key is one of: 'outputs','items','data','__list__'."""
    if isinstance(data, list):
        return data, '__list__'
    if isinstance(data, dict):
        for key in ('outputs', 'items', 'data'):
            val = data.get(key)
            if isinstance(val, list):
                return val, key
        # no obvious list; keep as single-record list
        return [data], '__list__'
    raise TypeError(f"Unsupported top-level JSON: {type(data).__name__}")

def get_prompt_text(rec: Dict[str, Any]) -> str:
    for k in ('question','prompt','text','instruction','input','content'):
        if k in rec and isinstance(rec[k], str) and rec[k].strip():
            return rec[k].strip()
    # Fallback: stringify the record if no known text field
    return json.dumps(rec, ensure_ascii=False)

def add_model_answer(rec: Dict[str, Any], answer: str) -> Dict[str, Any]:
    out = dict(rec)
    out['model_answer'] = answer
    return out

def write_preserving_top(data_original: Any, records_out: List[Dict[str, Any]], container_key: str, path_out: Path):
    if container_key == '__list__':
        to_write = records_out
    else:
        to_write = dict(data_original)
        to_write[container_key] = records_out
    path_out.parent.mkdir(parents=True, exist_ok=True)
    path_out.write_text(json.dumps(to_write, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Wrote JSON to:', path_out)


In [ ]:
# 7) Prompt-answer cache (resume support)
# Answers are cached by prompt text, so an interrupted run re-reads completed
# calls from disk instead of paying for them again.

def load_cache(provider_name: str) -> Dict[str, str]:
    cpath = CACHE_DIR / f'{provider_name}_qa_cache.json'
    if cpath.exists():
        try:
            return json.loads(cpath.read_text(encoding='utf-8'))
        except Exception:
            return {}
    return {}

def save_cache(provider_name: str, cache: Dict[str, str]):
    cpath = CACHE_DIR / f'{provider_name}_qa_cache.json'
    cpath.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
# 8) Run Filler — simple counter + flat JSON output
# Options: 'openai', 'anthropic', 'gemini'
PROVIDER = 'openai'   # <--- Fill with one of three

OVERWRITE_EXISTING = True
MAX_TOKENS = 512

from pathlib import Path
import json, datetime as dt

# Provider (uses the globally defined SYSTEM_WRITER inside provider classes)
provider = get_provider(PROVIDER)
if not provider.available():
    raise RuntimeError(f"Provider '{PROVIDER}' not available — check API key/SDK in earlier cells.")

# Load prompts
data = read_json_like(PROMPTS_PATH)
rows, _ = peel_to_list(data)
items = [dict(r) if isinstance(r, dict) else {'question': str(r)} for r in rows]
N = len(items)

# Output paths: flat file next to the input
out_path = PROMPTS_PATH.with_name(f"{PROMPTS_PATH.stem}__{PROVIDER}__answered.json")

# Optional cache if earlier cells defined it; otherwise a no-op dict
try:
    cache = load_cache(PROVIDER)
    _save_cache = lambda: save_cache(PROVIDER, cache)
except NameError:
    cache = {}
    _save_cache = lambda: None

def _prompt_of(rec: dict) -> str:
    return get_prompt_text(rec)

def _rec_id(rec: dict, i: int) -> str:
    return rec.get("id") or rec.get("name") or rec.get("topic") or f"index_{i}"

# Generate with a visible counter and periodic saves
for i, rec in enumerate(items, start=1):
    pid = _rec_id(rec, i)
    print(f"[{i}/{N}] Generating: {pid}")

    if (not OVERWRITE_EXISTING) and isinstance(rec.get("model_answer"), str) and rec["model_answer"].strip():
        # keep existing
        pass
    else:
        prompt = _prompt_of(rec)
        if prompt in cache:
            rec["model_answer"] = cache[prompt]
        else:
            try:
                ans = provider.generate(prompt, temperature=0.0, max_tokens=MAX_TOKENS)
            except Exception as e:
                ans = f"[ERROR] {e}"
            rec["model_answer"] = ans
            cache[prompt] = ans
            _save_cache()

    # Lightweight periodic save (flat JSON list only)
    if (i % 25 == 0) or (i == N):
        out_path.write_text(json.dumps(items, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nSaved (flat JSON):", str(out_path))

